In [1]:
import torch
from torch import nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
splits = {'train': 'small/train-00000-of-00001.parquet', 'validation': 'small/validation-00000-of-00001.parquet', 'test': 'small/test-00000-of-00001.parquet'}
dataset = pd.read_parquet("hf://datasets/google/code_x_glue_cc_code_refinement/" + splits["train"])
dataset.head()

,id,buggy,fixed
0,0,public java.lang.String METHOD_1 ( ) { return ...,public java.lang.String METHOD_1 ( ) { return ...
1,1,public boolean METHOD_1 ( java.lang.String nam...,public boolean METHOD_1 ( java.lang.String nam...
2,2,public char METHOD_1 ( java.lang.String VAR_1 ...,public char METHOD_1 ( java.lang.String VAR_1 ...
3,3,private void METHOD_1 ( ) { VAR_1 . METHOD_2 (...,private void METHOD_1 ( ) { VAR_1 . METHOD_2 (...
4,4,public TYPE_1 METHOD_1 ( ) { java.lang.System....,public TYPE_1 METHOD_1 ( ) { return this . VAR...


In [3]:
dataset = dataset[:10000]
len(dataset)

10000

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [18]:
from torch.utils.data import Dataset,DataLoader,random_split

class CodeDataset(Dataset):
    def __init__(self,data,max_tokens):
        self.x = data["buggy"]
        self.x = list(map(self.tokensize,self.x))
        temp = np.zeros((len(self.x),max_tokens))
        for i in range(len(temp)):
            temp[i,:min(len(self.x[i]),max_tokens)] = np.array(self.x[i])
        self.x = torch.LongTensor(temp)
        self.y = data["fixed"]
        self.y = list(map(self.tokensize,self.y))
        temp = np.zeros((len(self.y),max_tokens))
        for i in range(len(temp)):
            temp[i,:min(len(self.y[i]),max_tokens)] = np.array(self.y[i])
        self.y = torch.LongTensor(temp)

    def __len__(self):
        return len(self.x)

    keywords = ['abstract', 'continue', 'for', 'new', 'switch', 'assert', 'default', 'goto', 'package', 'synchronized', 'boolean', 'do', 'if', 
                'private', 'this', 'break', 'double', 'implements', 'protected', 'throw', 'byte', 'else', 'import', 'public', 'throws', 'case', 
                'enum', 'instanceof', 'return', 'transient', 'catch', 'extends', 'int', 'short', 'try', 'char', 'final', 'interface', 'static', 
                'void', 'class', 'finally', 'long', 'strictfp', 'volatile', 'const', 'float', 'native', 'super', 'while','instanceof','java',
                'lang','String','METHOD','[END]','[START]']

    operator = []


    def tokensize(self,x):
        j = -1
        tokens = [128+self.keywords.index('[START]')]
        for i in range(len(x)):
            if x[i].isalpha():
                if j == -1:
                    j = i
            else:
                if j != -1:
                    if x[j:i] in self.keywords:
                        tokens.append(128+self.keywords.index(x[j:i]))
                    else:
                        tokens.extend(list(map(ord,x[j:i])))
                    j = -1
                if x[i] == ' ':
                    tokens.append(0)
                else:
                    tokens.append(ord(x[i]))
        tokens.append(128+len(self.keywords)-1)
        return tokens
    @staticmethod
    def getVocab():
        return 128+len(CodeDataset.keywords)
    def __getitem__(self,idx):
        return self.x[idx],self.y[idx]

In [19]:
dataset_trh = CodeDataset(dataset,400)
valid_ratio = .1
test_ratio = .1

test_size = int(test_ratio*len(dataset_trh))
valid_size = int(valid_ratio*len(dataset_trh))
training_size = len(dataset_trh) - test_size - valid_size


(train_data, test_data, valid_data) = random_split(dataset_trh,[training_size,test_size,valid_size])

In [20]:
class Embedder(nn.Module):
    def __init__(self,vocab,embemding_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab,embemding_dim)
    def forward(self,x):
        return self.emb(x)

In [40]:
class Attention(nn.Module):
    def __init__(self,size,size2):
        super().__init__()
        self.W_Q = nn.Parameter(torch.randn((size,size2)))
        self.W_K = nn.Parameter(torch.randn((size,size2)))
        self.W_V = nn.Parameter(torch.randn((size,size2)))
        self.sf = nn.Softmax(dim=-1)
        self.size2 = size2

    def forward(self,x):
        return self.sf(1/np.sqrt(self.size2)*((x@self.W_Q)@(x@self.W_V).mT))@x@self.W_V

class AddAndNormalise(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self,x,y):
        h = torch.concat([x,y],dim = -1)
        return (h - h.mean())/h.std()


class MultiHeadAttention(nn.Module):
    def __init__(self,nheads,input_dim,attention_layers,device):
        super().__init__()
        self.device = device
        self.W_Q = nn.Parameter(torch.randn((input_dim,attention_layers*nheads)))
        self.W_K = nn.Parameter(torch.randn((input_dim,attention_layers*nheads)))
        self.W_V = nn.Parameter(torch.randn((input_dim,attention_layers*nheads)))
        self.W_O = nn.Parameter(torch.randn((attention_layers*nheads,attention_layers*nheads)))

        self.sf = nn.Softmax(dim=-1)
        self.attention_layers = attention_layers
        self.nheads = nheads

    def forward(self,x):
        batchsize, input_len = x.shape[0],x.shape[1]
        Q = x@self.W_Q
        K = x@self.W_K
        V = x@self.W_V

        """
        previous attempt
        Qs = torch.chunk(Q,self.nheads,dim = -1)
        Ks = torch.chunk(K,self.nheads,dim = -1)
        Vs = torch.chunk(V,self.nheads,dim = -1)
        c = torch.concat([self.scaledDotProduct(Qs[i],Ks[i],Vs[i]) for i in range(self.nheads)],dim=-1)
        return c@self.W_O
        """

        Qs = Q.reshape(batchsize,input_len,self.nheads,self.attention_layers).transpose(1,2)
        Ks = K.reshape(batchsize,input_len,self.nheads,self.attention_layers).transpose(1,2)
        Vs = V.reshape(batchsize,input_len,self.nheads,self.attention_layers).transpose(1,2)

        out = self.scaledDotProduct(Qs,Ks,Vs).transpose(1,2).reshape(batchsize,input_len,-1)
        return out@self.W_O


    def scaledDotProduct(self,Q,K,V):
        return self.sf(1/torch.sqrt(torch.Tensor([Q.shape[-1]]).to(self.device))*(Q@(K.mT)))@V

class FeedForwardLayer(nn.Module):
    def __init__(self,input_dim,hidden_dim,output_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim,hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim,output_dim)
        )
    def forward(self,x):
        return self.model(x)

class Encoder(torch.nn.Module):
    def __init__(self,input_dim,heads,attention_dim,hidden_dim,output_dim,device):
        super().__init__()
        self.device = device
        self.input_dim = input_dim
        if attention_dim % heads != 0:
            raise ArithmeticError(f"take a divisible number for heads , heads{heads} , attention_dim{attention_dim}")
        self.att = MultiHeadAttention(heads,input_dim+1,attention_dim//heads,device)
        self.norm = AddAndNormalise()
        self.W_1 = nn.Parameter(torch.randn((attention_dim+input_dim,hidden_dim)))
        self.b_1 = nn.Parameter(torch.randn(hidden_dim))
        self.W_2 = nn.Parameter(torch.randn((hidden_dim,output_dim)))
        self.b_2 = nn.Parameter(torch.randn(output_dim))
        self.r = nn.ReLU()
        self.ff = lambda x : self.r(x@self.W_1+self.b_1)@self.W_2 + self.b_2

    def forward(self,x):
        x2 = self.position_embedded(x)
        x3 = self.att(x2)
        #print(x3.shape)
        x4 = self.norm(x2,x3)

        return self.ff(x4)

    def position_embedded(self,x):
        P = torch.zeros([x.shape[0],x.shape[1],1]).to(self.device)
        for i in range(x.shape[1]):
            P[:,i,0] = 2**(-i)
        return torch.concat([x,P], dim = -1)

In [41]:
class Decoder(torch.nn.Module):
    def __init__(self,input_dim,heads,attention_dim,hidden_dim,embedding_dim,output_size,device):
        """
        masked att
        add norm
        att
        add norm
        feed forward
        add norm

        linear
        softmax
        """
        super().__init__()
        self.device = device
        self.output_size = output_size
        self.embedding_dim = embedding_dim
        if attention_dim % heads != 0:
            raise ArithmeticError(f"take a divisible number for heads , heads{heads} , attention_dim{attention_dim}")
        self.att = MultiHeadAttention(heads,embedding_dim,attention_dim//heads,device)
        self.att2 = MultiHeadAttention(heads,embedding_dim+attention_dim+embedding_dim,attention_dim//heads,device)
        self.norm = AddAndNormalise()
        # self.W_1 = nn.Parameter(torch.randn((attention_dim+input_dim,hidden_dim)))
        # self.b_1 = nn.Parameter(torch.randn(hidden_dim))
        # self.W_2 = nn.Parameter(torch.randn((hidden_dim,output_dim)))
        # self.b_2 = nn.Parameter(torch.randn(output_dim))
        # self.r = nn.ReLU()
        # self.ff = lambda x : self.r(x@self.W_1+self.b_1)@self.W_2 + self.b_2
        self.ff = FeedForwardLayer(embedding_dim+attention_dim+embedding_dim+attention_dim, hidden_dim,embedding_dim)

    def forward(self,x,y):
        #y = torch.zeros((x.shape[0],self.output_size,self.embedding_dim))

        for i in range(output_size):
            x2 = self.att(y[:,0:i+1,:])
            x3 = self.norm(x2,y[:,0:i+1,:])
            x3 = torch.concat([x,x3],dim=-1)
            x4 = self.att2(x3)
            x5 = self.norm(x4,x3)
            x6 = self.ff(x5)
            y[:,i+1,:] = x6

        return y

In [42]:
class EncoderDecoder(nn.Module):
    def __init__(self,vocab,embedding_dim,output_size,device):
        super().__init__()
        self.emb = Embedder(vocab,embedding_dim)
        self.encoder = Encoder(embedding_dim,8,512,512,embedding_dim,device)
        self.decoder = Decoder(embedding_dim,8,512,512,embedding_dim,output_size,device)
        self.sm = nn.Sequential(
            nn.Linear(embedding_dim,vocab),
            nn.Softmax(dim=-1)
        )
        self.createy = lambda x : self.startv(x,output_size,embedding_dim)
    def startv(self,x,output_size,embedding_dim):
        y = torch.zeros((x,output_size,embedding_dim))
        y[:,0,:] = self.emb(torch.Tensor([[128+CodeDataset.getVocab()-1]]))
        return y
    def forward(self,x):
        x2 = self.emb(x)
        x3 = self.encoder(x2)
        x4 = self.decoder(x3,self.createy(x.shape[0]))
        x5 = self.sm(x4)
        return x5

In [43]:
train_dataset = DataLoader(train_data,128)
valid_dataset = DataLoader(valid_data,2048)
test_dataset = DataLoader(test_data,2048)
training_size

8000

In [44]:
import gc
gc.collect()
torch.cuda.empty_cache()

model = EncoderDecoder(CodeDataset.getVocab(),128,400,device)
model.to(device)
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

opt = torch.optim.Adam(model.parameters(),lr = 1e-4)

max_epochs = 100
model.train()
for i in range(max_epochs):
    s = 0
    for (x,y) in train_dataset:
        x = x.to(device)
        y = y.to(device)
        opt.zero_grad()
        pred = model(x).transpose(1,2)
        loss = loss_fn(pred,y)
        loss.backward()
        s+=loss.item()
        opt.step()
    if (i%5 == 0):
        with torch.no_grad():
            total = 0
            n = 0
            for (x,y) in valid_dataset:
                x = x.to(device)
                y = y.to(device).reshape(-1)
                pred = model(x)
                pred = torch.argmax(pred,dim = -1).reshape(-1)
                total += (pred == y).sum() - (0 == y).sum()
                n += len(y)- (0 == y).sum()
        print(total/n)

OutOfMemoryError: CUDA out of memory. Tried to allocate 100.00 MiB. GPU 0 has a total capacity of 3.64 GiB of which 94.25 MiB is free. Process 3778 has 460.78 MiB memory in use. Process 32131 has 39.72 MiB memory in use. Process 32266 has 29.97 MiB memory in use. Process 32518 has 190.65 MiB memory in use. Process 56874 has 1.99 GiB memory in use. Including non-PyTorch memory, this process has 516.00 MiB memory in use. Process 78726 has 21.72 MiB memory in use. Of the allocated memory 414.18 MiB is allocated by PyTorch, and 17.82 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)